# IDX Stock Forecasting with TFT (Kaggle Runner)

Notebook ini dirancang untuk menjalankan pelatihan, evaluasi, studi ablasi, dan inferensi model **Temporal Fusion Transformer (TFT)** pada platform Kaggle. 

Notebook ini secara otomatis mendukung pelatihan terdistribusi multi-GPU berbasis **PyTorch DistributedDataParallel (DDP)** jika Anda mengaktifkan akselerator multi-GPU di Kaggle (seperti Dual T4 atau A100).

### 1. Inisialisasi Workspace & Git Clone
Sel ini digunakan untuk menghapus folder lama (jika ada) di direktori kerja Kaggle (`/kaggle/working`), melakukan cloning repository langsung dari GitHub, dan berpindah ke direktori proyek utama.

In [ ]:
# Hapus direktori lama jika ada untuk menghindari konflik
!rm -rf forecast

# Clone repository dari GitHub
!git clone https://github.com/MaulanaSeto/forecast.git

# Pindah ke direktori proyek
%cd forecast

# Setup system path
import os
import sys
sys.path.append(os.getcwd())
print("Direktori kerja aktif:", os.getcwd())

### 1b. Kustomisasi Konfigurasi (config.yaml)
Anda dapat mengedit parameter konfigurasi proyek (`config.yaml`) secara programatik menggunakan Python. Cara ini memudahkan Anda untuk menimpa nilai variabel hyperparameter tertentu satu per satu sebelum pelatihan dijalankan.

In [ ]:
import yaml

# 1. Muat file konfigurasi aktif
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# 2. Ubah hyperparameter sesuai preferensi Anda di sini
config['lr'] = 0.001          # Learning Rate
config['batch_size'] = 16    # Ukuran Batch (aman dari OOM dengan gradient checkpointing)
config['max_epochs'] = 30     # Batas Epoch Maksimum
config['max_gpus'] = 2        # Batas GPU Maksimum untuk DDP
config['use_checkpoint'] = True # Aktifkan Gradient Checkpointing untuk menghemat memori GPU
config['num_workers'] = 2     # Worker Multi-processing DataLoader (maksimal 2 untuk Kaggle CPU)

# 3. Simpan kembali perubahan
with open('config.yaml', 'w') as f:
    yaml.safe_dump(config, f)

print("Berhasil memperbarui config.yaml!")
print("Konfigurasi aktif saat ini:")
print(yaml.dump(config))

### 1c. Setup Path Dataset Kaggle
Karena data kompetisi tidak di-push ke repositori GitHub, Anda harus menambahkannya sebagai input dataset pada notebook Kaggle Anda. 

Sel di bawah ini mengarahkan `data_dir` di `config.yaml` ke direktori resmi kompetisi di Kaggle input. Jika nama kompetisi berbeda, sel ini akan otomatis memindai seluruh direktori `/kaggle/input/` secara rekursif untuk mendeteksi lokasi file `train.csv` secara dinamis.

In [ ]:
import yaml

# Path dataset resmi kompetisi di Kaggle
kaggle_dataset_dir = '/kaggle/input/competitions/idx-massive-multi-asset-forecasting-dl-genap-2025-2026'

# Fallback scan jika path resmi di atas tidak terdeteksi
if not os.path.exists(kaggle_dataset_dir):
    kaggle_input_dir = '/kaggle/input'
    if os.path.exists(kaggle_input_dir):
        for root, dirs, files in os.walk(kaggle_input_dir):
            if 'train.csv' in files:
                kaggle_dataset_dir = root
                break

if os.path.exists(kaggle_dataset_dir) and 'train.csv' in os.listdir(kaggle_dataset_dir):
    print(f"Dataset terdeteksi di: {kaggle_dataset_dir}")
    
    # Muat config dan update data_dir
    with open('config.yaml', 'r') as f:
        config = yaml.safe_load(f)
        
    config['data_dir'] = kaggle_dataset_dir
    
    with open('config.yaml', 'w') as f:
        yaml.safe_dump(config, f)
        
    print("[OK] Path 'data_dir' di config.yaml berhasil diarahkan ke:", kaggle_dataset_dir)
else:
    print("[WARNING] Dataset 'train.csv' tidak terdeteksi di /kaggle/input.")
    print("Silakan klik '+ Add Input' di pojok kanan Kaggle, cari dataset kompetisi Anda, lalu jalankan ulang sel ini.")

### 2. Pemeriksaan Akselerator GPU
Mari periksa ketersediaan GPU di notebook Kaggle Anda.

In [ ]:
import torch

available_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Total GPU Terdeteksi: {available_gpus}")
for i in range(available_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

### 3. Instalasi Dependensi
Instal paket-paket Python yang dibutuhkan oleh proyek ini.

> **Catatan Penting untuk Kaggle**: Kaggle sudah menyediakan pustaka utama (PyTorch, Pandas, NumPy, Scikit-learn, dll.) secara bawaan. Anda dapat langsung menjalankan training tanpa menginstal ulang. Jika Anda tetap menjalankan sel di bawah ini, **abaikan saja jika muncul pesan peringatan/error konflik dependensi** dari pustaka RAPIDS (`cuml-cu12`, `cudf-cu12`, dll.) karena pustaka tersebut tidak digunakan dalam proyek ini.

In [ ]:
# Install dependensi proyek
# Anda bisa menambahkan bendera --no-deps jika ingin menghindari konflik resolusi paket bawaan Kaggle
!pip install -r requirements.txt --no-deps

### 4. Eksekusi Pelatihan Model (Full Pipeline)
Menjalankan training dan evaluasi model TFT baseline. Program akan otomatis mendeteksi ketersediaan GPU dan memicu pelatihan multi-process DDP secara paralel apabila terdeteksi $>1$ GPU.

In [ ]:
# Mode full akan menjalankan:
# 1. Preprocessing data
# 2. Training TFT Model (DDP Multi-GPU)
# 3. Evaluasi & Inferensi Test Set
# 4. Pembuatan File Submission
!python main.py --mode full

### 5. Pengujian Logika Model (Logical & Numerical Debugging)
Sebelum menjalankan model penuh pada seluruh dataset, jalankan modul tes logika berikut untuk memverifikasi fungsionalitas matematika, kausalitas waktu, dan isolasi saham:

In [ ]:
# 1. Tes Overfitting Satu Batch (Memastikan model bisa belajar & loss turun ke ~0)
!python main.py --mode full --debug --debug-mode overfit

In [ ]:
# 2. Tes Kausalitas Temporal (Memastikan tidak ada kebocoran data masa depan / lookahead bias)
!python main.py --mode full --debug --debug-mode causality

In [ ]:
# 3. Tes Permutasi Saham (Memastikan urutan ticker bersifat independen / tidak ada kebocoran antar saham)
!python main.py --mode full --debug --debug-mode permutation

### 6. Visualisasi Metrik & Grafik Hasil Pelatihan
Menampilkan kurva kerugian (*training loss*) dan validation RMSE per epoch dari sesi pelatihan terbaru.

In [ ]:
import pandas as pd
from IPython.display import Image, display

checkpoint_dir = 'checkpoints'
if os.path.exists(checkpoint_dir):
    runs = [os.path.join(checkpoint_dir, d) for d in os.listdir(checkpoint_dir) if d.startswith('run_')]
    if runs:
        latest_run = max(runs, key=os.path.getmtime)
        print(f"Menampilkan hasil dari sesi terbaru: {latest_run}")
        
        # Tampilkan grafik training
        history_plot = os.path.join(latest_run, 'history.png')
        if os.path.exists(history_plot):
            display(Image(filename=history_plot))
            
        # Tampilkan 10 epoch terakhir dari CSV
        metrics_csv = os.path.join(latest_run, 'metrics.csv')
        if os.path.exists(metrics_csv):
            df = pd.read_csv(metrics_csv)
            display(df.tail(10))
    else:
        print("Belum ada folder sesi training (run_*) yang terdeteksi.")
else:
    print("Folder checkpoints belum dibuat.")

### 7. Studi Ablasi (Ablation Study)
Menjalankan studi komparasi dengan memotong komponen Variable Selection Network (VSN) dan Temporal Self-Attention untuk mengukur signifikansi performanya.

In [ ]:
# Jalankan studi ablasi
!python main.py --mode ablation

### 8. Hasil Studi Ablasi
Menampilkan plot perbandingan performa model baseline dengan varian model ablatif.

In [ ]:
ablation_dirs = []
for root, dirs, files in os.walk('checkpoints'):
    if 'ablation' in dirs:
        ablation_dirs.append(os.path.join(root, 'ablation'))

if ablation_dirs:
    latest_ablation = max(ablation_dirs, key=os.path.getmtime)
    print(f"Menampilkan hasil studi ablasi dari: {latest_ablation}")
    
    # Tampilkan grafik perbandingan
    comp_plot = os.path.join(latest_ablation, 'ablation_comparison.png')
    if os.path.exists(comp_plot):
        display(Image(filename=comp_plot))
        
    # Tampilkan summary JSON
    results_json = os.path.join(latest_ablation, 'ablation_results.json')
    if os.path.exists(results_json):
        import json
        with open(results_json, 'r') as f:
            print(json.dumps(json.load(f), indent=2))
else:
    print("Belum ada sesi studi ablasi yang dijalankan.")

### 9. Verifikasi File Submission
Sel terakhir ini memastikan bahwa file submisi `results/tft.csv` telah berhasil dibuat dengan format baris yang sesuai.

In [ ]:
submission_path = 'results/tft.csv'
if os.path.exists(submission_path):
    sub_df = pd.read_csv(submission_path)
    print(f"[OK] File submission terdeteksi di {submission_path}")
    print(f"Total Baris Data: {len(sub_df):,}")
    display(sub_df.head(10))
else:
    print("[ERROR] File submission tft.csv tidak ditemukan di folder results.")